<a href="https://colab.research.google.com/github/fonglieu/text-analytics-assignment5/blob/main/Assignment_5_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 5 — Option B: Job Fit Analyzer
## BSAN 6200: Text Mining & Social Media Analytics — Spring 2026

**Student Name:** Fong Lieu
**Option:** B — Job Fit Analyzer  
**API Path:** [Paid / Free]  

---

### Table of Contents
1. [Setup and Imports](#1-setup)
2. [Load Job Descriptions and Resume](#2-loading)
3. [Text Chunking](#3-chunking)
4. [Embedding and Vector Store](#4-embedding)
5. [Analysis Prompts and Chain](#5-analysis)
6. [Zero-shot vs. Few-shot Comparison](#6-comparison)
7. [Evaluation](#7-evaluation)

> **Reminder:** The Streamlit app is a separate file (`streamlit_app.py`). This notebook builds and tests the analysis pipeline.  
> See the Option B Implementation Guide for detailed step requirements.

---
<a id="1-setup"></a>
## 1. Setup and Imports

Install required packages and load your API key from a `.env` file.  
**Do NOT hardcode API keys in this notebook.**

Suggested packages: `langchain`, `langchain-openai` or `langchain-community`, `chromadb` or `faiss-cpu`, `pypdf`, `python-dotenv`, `pandas`, `sentence-transformers` (free path)

In [1]:
# ── Install packages (uncomment as needed) ──
!pip install openai pypdf pandas chromadb sentence-transformers -q

from google.colab import drive
drive.mount('/content/drive')

# ── Your imports below ──
import os
import pandas as pd
from openai import OpenAI
from google.colab import userdata

# ── API key ──
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# ── Paths ──
BASE_DIR = "/content/drive/MyDrive/data"

print("Imports successful.")
print(f"Files: {os.listdir(BASE_DIR)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Imports successful.
Files: ['Fong_Lieu_Resume.pdf', 'jd_08_capgemini_junior_ba.txt', 'jd_03_spotify_data_analyst.txt', 'jd_10_goldmansachs_data_office_analyst.txt', 'jd_09_bayview_data_ops_analyst.txt', 'jd_02_att_strategy_analyst.txt', 'jd_04_delta_data_analyst.txt', 'jd_06_transformlabs_business_analyst.txt', 'jd_metadata.csv', 'jd_05_northrop_grumman_data_analyst.txt', 'jd_01_amend_sales_analyst.txt', 'jd_07_qualifiedhealth_product_intern.txt', 'chroma_db']


---
<a id="2-loading"></a>
## 2. Load Job Descriptions and Resume

**Required:**
- 10+ JD files in `data/job_descriptions/` (each as a separate .txt or .pdf)
- Your resume in `data/resume/`
- A metadata file `data/jd_metadata.csv` with columns: filename, company, title, source_url, date_collected

Print: number of JDs loaded, number of resume docs, and preview content from each.

In [2]:
# ── Load JD metadata ──
metadata_df = pd.read_csv(os.path.join(BASE_DIR, "jd_metadata.csv"))
print(f"JDs loaded: {len(metadata_df)}\n")
display(metadata_df)

JDs loaded: 10



,filename,company,title,source_url,date_collected
0,jd_01_amend_sales_analyst.txt,AMEND Consulting,Sales Analyst – Co-op/Internship,https://job-boards.greenhouse.io/amendconsulti...,2026-05-09
1,jd_02_att_strategy_analyst.txt,AT&T,Strategy Analyst – Wireline Transformation & S...,https://www.att.jobs/job/-/strategy-analyst/11...,2026-05-09
2,jd_03_spotify_data_analyst.txt,Spotify,Data Analyst II – Performance Optimization Squad,https://jobs.lever.co/spotify/d95e1989-9a1a-48...,2026-05-09
3,jd_04_delta_data_analyst.txt,Delta Air Lines,Senior Analyst – Customer Data Solutions,https://delta.avature.net/en_US/careers/JobDet...,2026-05-09
4,jd_05_northrop_grumman_data_analyst.txt,Northrop Grumman,Data Insight Analyst,https://jobs.northropgrumman.com/careers/job/1...,2026-05-09
5,jd_06_transformlabs_business_analyst.txt,Transform Labs,Entry-Level Business Analyst,https://awhnet.bamboohr.com/careers/76,2026-05-09
6,jd_07_qualifiedhealth_product_intern.txt,Qualified Health,Product Intern,https://lmu.joinhandshake.com/jobs/10994147,2026-05-09
7,jd_08_capgemini_junior_ba.txt,Capgemini America Inc.,Junior Business Analyst,https://lmu.joinhandshake.com/job-search/10982553,2026-05-09
8,jd_09_bayview_data_ops_analyst.txt,Bayview Asset Management,Data Operations Analyst,https://lmu.joinhandshake.com/job-search/10974135,2026-05-09
9,jd_10_goldmansachs_data_office_analyst.txt,Goldman Sachs,Asset & Wealth Management Data Office Analyst,https://www.linkedin.com/jobs/view/4375782060/,2026-05-09


In [5]:
# ── Load JD documents ──
def load_text_file(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        return f.read()

jd_docs = {}
for _, row in metadata_df.iterrows():
    filepath = os.path.join(BASE_DIR, row["filename"])
    jd_docs[row["filename"]] = {
        "text":       load_text_file(filepath),
        "company":    row["company"],
        "title":      row["title"],
        "source_url": row["source_url"]
    }

print(f"JD documents loaded: {len(jd_docs)}")

JD documents loaded: 10


In [6]:
# ── Load Resume ──
from pypdf import PdfReader

def load_pdf(filepath):
    reader = PdfReader(filepath)
    return "\n".join([page.extract_text() for page in reader.pages if page.extract_text()])

resume_text = load_pdf(os.path.join(BASE_DIR, "Fong_Lieu_Resume.pdf"))
print(f"Resume loaded. Length: {len(resume_text)} chars")

Resume loaded. Length: 3438 chars


In [7]:
# ── Preview sample content ──
sample_key = list(jd_docs.keys())[0]
print(f"=== Sample JD: {sample_key} ===")
print(jd_docs[sample_key]["text"][:500])

print("\n=== Resume (first 500 chars) ===")
print(resume_text[:500])

=== Sample JD: jd_01_amend_sales_analyst.txt ===
Company: AMEND Consulting
Title: Sales Analyst – Co-op/Internship
Location: Cincinnati, OH
Source: https://job-boards.greenhouse.io/amendconsulting/jobs/4005828009
Date Collected: 2026-05-09

--- JOB DESCRIPTION ---

About AMEND:
AMEND is a management consulting firm based in Cincinnati, OH with areas of focus in operations, analytics, and technology. We are focused on strengthening the people, processes, and systems in organizations to generate a holistic transformation. Our three-tiered approa

=== Resume (first 500 chars) ===
Fong  Lieu
 
Los  Angeles,  CA   |   (720)  921-4111   |   fonglieu8@gmail.com   
TECHNICAL  SKILLS  Programming  &  Statistical  Languages:  SQL,  Python,  R  Libraries  &  Frameworks:  pandas,  NumPy,  matplotlib,  scikit-learn,  dplyr,  tidyr  Data  Analysis  &  Visualization:  Tableau,  Power  BI,  Excel  Data  Management:  Data  Extraction  &  Transformation,  Databases,  Salesforce  EDUCATION  Loyola  Marym

---
<a id="3-chunking"></a>
## 3. Text Chunking

Split your documents into chunks.  
**Required:** Try at least 2 chunking strategies, compare them quantitatively, and justify your final choice.

**Hint:** JDs often have natural sections (Requirements, Responsibilities, Qualifications). Consider whether your splitter respects these boundaries.

In [13]:
# Helper Function
def chunk_text_recursive(text, chunk_size=500, overlap=50):
    """Strategy 1: fixed-size sliding window chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

def chunk_text_by_section(text, chunk_size=800, overlap=100):
    """Strategy 2: split on double newlines (natural sections), then merge small ones."""
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks = []
    current = ""
    for para in paragraphs:
        if len(current) + len(para) <= chunk_size:
            current += "\n\n" + para
        else:
            if current:
                chunks.append(current.strip())
            current = para
    if current:
        chunks.append(current.strip())
    return chunks

In [14]:
# ── Strategy 1 ──
all_text_s1 = " ".join([d["text"] for d in jd_docs.values()]) + " " + resume_text
chunks_1 = chunk_text_recursive(all_text_s1, chunk_size=500, overlap=50)

print(f"Strategy 1 | Total chunks: {len(chunks_1)}")
print(f"Avg length: {sum(len(c) for c in chunks_1) / len(chunks_1):.0f} chars")
print(f"Min: {min(len(c) for c in chunks_1)} | Max: {max(len(c) for c in chunks_1)}")

Strategy 1 | Total chunks: 80
Avg length: 496 chars
Min: 155 | Max: 500


In [15]:
# ── Strategy 2 ──
chunks_2 = []
for doc in jd_docs.values():
    chunks_2.extend(chunk_text_by_section(doc["text"], chunk_size=800, overlap=100))
chunks_2.extend(chunk_text_by_section(resume_text, chunk_size=800, overlap=100))

print(f"Strategy 2 | Total chunks: {len(chunks_2)}")
print(f"Avg length: {sum(len(c) for c in chunks_2) / len(chunks_2):.0f} chars")
print(f"Min: {min(len(c) for c in chunks_2)} | Max: {max(len(c) for c in chunks_2)}")

Strategy 2 | Total chunks: 55
Avg length: 647 chars
Min: 166 | Max: 3436


In [16]:
# ── Compare strategies ──
comparison = pd.DataFrame({
    "Strategy": [
        "Strategy 1: Recursive Fixed-Size (500/50)",
        "Strategy 2: Section-Aware (800/100)"
    ],
    "Total Chunks": [len(chunks_1), len(chunks_2)],
    "Avg Length (chars)": [
        round(sum(len(c) for c in chunks_1) / len(chunks_1)),
        round(sum(len(c) for c in chunks_2) / len(chunks_2))
    ],
    "Min Chunk": [min(len(c) for c in chunks_1), min(len(c) for c in chunks_2)],
    "Max Chunk": [max(len(c) for c in chunks_1), max(len(c) for c in chunks_2)]
})
display(comparison)

print("\n=== Strategy 1 sample chunk ===")
print(chunks_1[3])
print("\n=== Strategy 2 sample chunk ===")
print(chunks_2[3])

,Strategy,Total Chunks,Avg Length (chars),Min Chunk,Max Chunk
0,Strategy 1: Recursive Fixed-Size (500/50),80,496,155,500
1,Strategy 2: Section-Aware (800/100),55,647,166,3436



=== Strategy 1 sample chunk ===
o client engagement
- Research and identify potential sales leads and market trends to support an active and organized pipeline for the sales team, using tools such as ZoomInfo and Hunter
- Ensure accurate entry and ongoing maintenance of sales activity data in Salesforce and other key systems
- Prepare weekly sales reports and analyze data through Excel manipulation to track key performance indicators (KPIs)
- Generate and elevate insights on ways to improve sales support processes, systems, an

=== Strategy 2 sample chunk ===
Qualifications / Requirements:
- Pursuit of a degree in Business, Marketing, Sales, or a related field
- Ability to build relationships and collaborate with team members
- Executive presence and confident communication skills
- Strong organizational skills with excellent attention to detail
- Ability to manage multiple projects and deadlines simultaneously
- Proactive mindset, with the ability to take initiative and prioritize in

### Chunking Decision

**Which strategy did you choose?**  
**Why?**  
**Final settings (chunk_size, overlap):**

---
<a id="4-embedding"></a>
## 4. Embedding and Vector Store

Embed your chunks and store them in a vector database (ChromaDB or FAISS).

**Paid path:** OpenAI `text-embedding-3-small`  
**Free path:** `sentence-transformers/all-MiniLM-L6-v2`

After creating the store, run a test similarity search to verify it works.

In [17]:
# ── Create embeddings and vector store ──
from sentence_transformers import SentenceTransformer
import chromadb

# Load embedding model (free, no API key needed)
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Use Strategy 2 chunks
final_chunks = chunks_2

# Create ChromaDB collection
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("job_fit_analyzer")

# Embed and store in batches
batch_size = 50
for i in range(0, len(final_chunks), batch_size):
    batch = final_chunks[i:i+batch_size]
    embeddings = embedder.encode(batch).tolist()
    collection.add(
        documents=batch,
        embeddings=embeddings,
        ids=[f"chunk_{i+j}" for j in range(len(batch))]
    )

print(f"Vector store created with {collection.count()} chunks.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector store created with 55 chunks.


In [18]:
# ── Verify: run a test similarity search ──
def similarity_search(query, n_results=4):
    query_embedding = embedder.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results
    )
    return results["documents"][0]

test_query = "SQL Python data analysis dashboard experience required"
results = similarity_search(test_query)

print(f"Query: '{test_query}'\n")
for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(doc[:300])
    print()

Query: 'SQL Python data analysis dashboard experience required'

--- Result 1 ---
Responsibilities:
- Develop and maintain Tableau dashboards and data visualizations to support operational and strategic decisions
- Use SQL to query data from various systems and interpret complex datasets to generate clear, strategic insights
- Analyze and synthesize data in support of business pl

--- Result 2 ---
Qualifications / Requirements:
- Bachelor's degree in Data Science, Business Analytics, Statistics, Computer Science, or a related field
- 1+ years of significant work experience in a Business Analysis, Data Analytics, or Data Creation role
- Proficiency in data querying and modeling; ability to wri

--- Result 3 ---
Qualifications / Requirements:
- 2+ years of experience as a data analyst in a technical environment
- Strong SQL skills — comfortable writing complex queries from scratch
- Proficiency in Python for data manipulation, pipeline creation, and automation
- Experience with infrastru

---
<a id="5-analysis"></a>
## 5. Analysis Prompts and Chain

Build 3 analysis types, each with its own prompt:

1. **Skill Gap Report:** Compare resume skills vs. JD requirements. Output matching skills, missing skills, and recommended actions.
2. **Keyword Alignment:** Extract key terms from a JD, check which appear in the resume, report a match rate.
3. **Fit Summary:** 3-4 sentence narrative assessment citing evidence from both documents.

You also need to wire up the LLM and a way to pass a specific JD + resume into each prompt.

**Required:** Document at least 3 prompt iterations total (across any analysis type) with rationale.

**Reminder:** Prompt design must be your own work (Tier 2 — AI prohibited for this step).

In [19]:
# ── Initialize LLM ──
def call_llm(prompt, model="gpt-4o-mini", temperature=0):
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [20]:
# Helper Function
def get_jd_text(filename):
    return jd_docs[filename]["text"]

In [21]:
# ── Analysis 1: Skill Gap Report ──
def skill_gap_report(jd_text, resume_text):
    prompt = f"""
You are a career coach analyzing a candidate's resume against a job description.

JOB DESCRIPTION:
{jd_text}

CANDIDATE RESUME:
{resume_text}

Produce a structured Skill Gap Report with exactly three sections:

MATCHING SKILLS:
- List each skill or experience the candidate has that directly matches a JD requirement.
- Be specific, citing evidence from the resume (tool name, project, or role).

MISSING SKILLS:
- List each requirement from the JD that is absent or insufficiently demonstrated in the resume.
- Be specific about what is missing.

RECOMMENDED ACTIONS:
- For each missing skill, provide one concrete, actionable step the candidate can take
  (e.g., specific course, project idea, certification, or tool to learn).

Use bullet points. Be concise and factual. Do not invent skills not present in either document.
"""
    return call_llm(prompt)

# Test it
target_jd = get_jd_text("jd_10_goldmansachs_data_office_analyst.txt")
result = skill_gap_report(target_jd, resume_text)
print("=== SKILL GAP REPORT: Goldman Sachs ===\n")
print(result)

=== SKILL GAP REPORT: Goldman Sachs ===

### SKILL GAP REPORT

#### MATCHING SKILLS:
- **Data Analysis Experience**: The candidate has experience in data extraction and transformation as a Pricing Analyst at Onboard Logistics, which aligns with the requirement for data analysis and operations experience.
- **Technical Skills**: Proficiency in SQL and Python is demonstrated in the resume, matching the job's requirement for these programming languages.
- **Data Visualization**: The candidate has developed KPI dashboards in Tableau, which supports the need for preparing dashboards and reports for internal stakeholders.
- **Attention to Detail**: The candidate's work as a Pricing Analyst involved improving win rates and generating profit through data-driven strategies, indicating strong analytical skills and attention to detail.
- **Project Support**: Experience in delivering lessons and improving learning outcomes as a Professional Learning Facilitator shows the ability to track actions a

In [22]:
# ── Analysis 2: Keyword Alignment ──
def keyword_alignment(jd_text, resume_text):
    prompt = f"""
You are a resume keyword analyst.

JOB DESCRIPTION:
{jd_text}

CANDIDATE RESUME:
{resume_text}

Step 1 — Extract the 15 most important keywords and phrases from the job description
(focus on required skills, tools, technologies, and domain terms).

Step 2 — For each keyword, determine if it appears or is semantically equivalent in the resume.
Mark each as: MATCH, PARTIAL MATCH, or MISSING.

Step 3 — Calculate: match rate = (MATCH + 0.5 * PARTIAL MATCH) / 15

Output format:

KEYWORD TABLE:
| Keyword | Status | Evidence from Resume |
|---------|--------|----------------------|
...

MATCH RATE: X%

SUMMARY: One sentence interpreting the match rate and what it means for this application.
"""
    return call_llm(prompt)

result = keyword_alignment(target_jd, resume_text)
print("=== KEYWORD ALIGNMENT: Goldman Sachs ===\n")
print(result)

=== KEYWORD ALIGNMENT: Goldman Sachs ===

### KEYWORD TABLE:
| Keyword                                      | Status        | Evidence from Resume                                                                 |
|----------------------------------------------|---------------|--------------------------------------------------------------------------------------|
| Data governance                              | MISSING       | No mention of data governance in the resume.                                        |
| Data quality                                 | PARTIAL MATCH | Mention of data accuracy and completeness in the context of data validation checks.  |
| KPIs                                         | MATCH         | Developed KPI dashboards in Tableau for monitoring performance.                      |
| Data analysis                                 | MATCH         | Experience in data extraction, transformation, and analysis in various roles.       |
| SQL                       

In [23]:
# ── Analysis 3: Fit Summary ──
def fit_summary(jd_text, resume_text):
    prompt = f"""
You are a hiring consultant writing a brief fit assessment.

JOB DESCRIPTION:
{jd_text}

CANDIDATE RESUME:
{resume_text}

Write a 3-4 sentence narrative Fit Summary assessing how well this candidate fits the role.

- Sentence 1: State overall fit level (Strong / Moderate / Weak) and primary reason,
  citing specific evidence from both documents.
- Sentence 2: Highlight the candidate's single strongest relevant qualification for this role.
- Sentence 3: Identify the most significant gap between the candidate and the role's requirements.
- Sentence 4 (optional): One specific recommendation to strengthen the application.

Be direct and evidence-based. Do not use filler phrases.
Only reference information present in the documents provided.
"""
    return call_llm(prompt)

result = fit_summary(target_jd, resume_text)
print("=== FIT SUMMARY: Goldman Sachs ===\n")
print(result)

=== FIT SUMMARY: Goldman Sachs ===

**Fit Summary:** The candidate is a strong fit for the Analyst role in the XIG Data Office at Goldman Sachs due to their solid background in data analysis and experience with relevant tools such as SQL, Python, and Tableau, which align well with the job's technical requirements. Their experience as a Pricing Analyst, where they developed KPI dashboards and performed data extraction and transformation, demonstrates their ability to support data governance and quality initiatives effectively. However, the candidate lacks direct experience in private markets or alternative investments, which is preferred for this role. To strengthen their application, the candidate should emphasize any relevant coursework or projects related to asset management or private markets.


### Prompt Iteration Log

Document at least 3 total iterations across any of the analysis types.

**Iteration 1:** [Which analysis? What changed? Why? What improved?]

**Iteration 2:** [Which analysis? What changed? Why? What improved?]

**Iteration 3:** [Which analysis? What changed? Why? What improved?]

---
<a id="6-comparison"></a>
## 6. Zero-shot vs. Few-shot Comparison

Pick one of your 3 analysis types. Create a few-shot version by adding 1-2 example input/output pairs to the prompt. Run both versions on the same JD and compare outputs.

**Reminder:** You must write the few-shot examples yourself (Tier 2).

In [24]:
# ── Few-shot version of your chosen analysis ──
def skill_gap_report_fewshot(jd_text, resume_text):
    prompt = f"""
You are a career coach analyzing a candidate's resume against a job description.
Below is one example of the correct output format, followed by the actual task.

--- EXAMPLE ---
MATCHING SKILLS:
- SQL: Candidate lists SQL under Technical Skills and used it for data extraction at Onboard Logistics.
- Data Visualization: Candidate built Tableau dashboards for KPI reporting in current role.

MISSING SKILLS:
- dbt: JD requires dbt for pipeline management; not mentioned anywhere in the resume.
- A/B Testing: JD lists experimentation design as required; resume has no mention of this.

RECOMMENDED ACTIONS:
- dbt: Complete the free "dbt Fundamentals" course at courses.getdbt.com and build a personal project.
- A/B Testing: Take a Coursera or DataCamp course on A/B testing and apply it to a Kaggle dataset.
--- END EXAMPLE ---

Now perform the same analysis for the following:

JOB DESCRIPTION:
{jd_text}

CANDIDATE RESUME:
{resume_text}

MATCHING SKILLS:
MISSING SKILLS:
RECOMMENDED ACTIONS:
"""
    return call_llm(prompt)

In [25]:
# ── Run both on the same JD, display side by side ──
comparison_jd = get_jd_text("jd_09_bayview_data_ops_analyst.txt")

zero_shot_output = skill_gap_report(comparison_jd, resume_text)
few_shot_output  = skill_gap_report_fewshot(comparison_jd, resume_text)

print("=" * 60)
print("ZERO-SHOT OUTPUT — Bayview Asset Management")
print("=" * 60)
print(zero_shot_output)

print("\n" + "=" * 60)
print("FEW-SHOT OUTPUT — Bayview Asset Management")
print("=" * 60)
print(few_shot_output)

ZERO-SHOT OUTPUT — Bayview Asset Management
### SKILL GAP REPORT

#### MATCHING SKILLS:
- **SQL**: The candidate lists SQL as a technical skill and has experience in data extraction and transformation, which aligns with the JD's requirement for proficiency in SQL.
- **Python**: The candidate has Python listed as a programming language and has utilized it in projects, such as the Gender Pay Inequity Analysis, demonstrating analytical capabilities.
- **Tableau**: The candidate has experience developing KPI dashboards in Tableau, which matches the JD's preference for experience with data visualization tools.
- **Data Extraction & Transformation**: The candidate's role as a Pricing Analyst involved data extraction and transformation, aligning with the JD's requirement for ETL processes.
- **Analytical Skills**: The candidate has demonstrated strong analytical skills through various projects and work experiences, such as analyzing freight rates and conducting sentiment analysis.
- **Communi

### Zero-shot vs. Few-shot Analysis

**Which analysis type did you compare?**

**Which performed better?**

**Why? (use specific examples from the outputs above)**

---
<a id="7-evaluation"></a>
## 7. Evaluation

Run all 3 analysis types on your **top 3 target JDs** (9 total analyses).

For each, score:
- **Retrieval relevance:** Did it pull the right JD sections? (Yes/Partial/No)
- **Skill identification accuracy:** Are identified skills/gaps correct? (count correct vs. incorrect)
- **Actionability:** Are recommendations specific and useful? (1-5)
- **Faithfulness:** Does output stick to document content? (Faithful/Partial/Hallucinated)

**Reminder:** Evaluation must be your own work (Tier 2 — AI prohibited).

In [27]:
# ── Run 9 analyses (3 JDs x 3 analysis types) ──
top_3_jds = [
    ("jd_10_goldmansachs_data_office_analyst.txt", "Goldman Sachs"),
    ("jd_09_bayview_data_ops_analyst.txt",          "Bayview Asset Management"),
    ("jd_04_delta_data_analyst.txt",                "Delta Air Lines"),
]

results_store = {}

for filename, company in top_3_jds:
    jd_text = get_jd_text(filename)
    results_store[company] = {
        "skill_gap":         skill_gap_report(jd_text, resume_text),
        "keyword_alignment": keyword_alignment(jd_text, resume_text),
        "fit_summary":       fit_summary(jd_text, resume_text)
    }
    print(f"Completed: {company}")

print("\nAll 9 analyses complete.")

Completed: Goldman Sachs
Completed: Bayview Asset Management
Completed: Delta Air Lines

All 9 analyses complete.


In [28]:
# Output
for company, analyses in results_store.items():
    print("=" * 70)
    print(f"COMPANY: {company}")
    print("=" * 70)
    for analysis_type, output in analyses.items():
        print(f"\n--- {analysis_type.upper().replace('_', ' ')} ---")
        print(output)
    print()

COMPANY: Goldman Sachs

--- SKILL GAP ---
### SKILL GAP REPORT

#### MATCHING SKILLS:
- **Data Analysis Experience**: The candidate has experience in data extraction and transformation as a Pricing Analyst at Onboard Logistics, which aligns with the requirement for data analysis and operations experience.
- **Technical Skills**: Proficiency in SQL and Python is demonstrated in the resume, matching the job's requirement for these programming languages.
- **Data Visualization**: The candidate has developed KPI dashboards in Tableau, which corresponds to the need for preparing dashboards and reports for internal stakeholders.
- **Attention to Detail**: The candidate's work in improving win rates and generating gross profit through data-driven strategies indicates strong analytical skills and attention to detail.
- **Project Support**: Experience in tracking actions and timelines as a Professional Learning Facilitator shows capability in supporting project delivery.
- **Stakeholder Engagem

In [29]:
# ── Summarize evaluation results ──
eval_data = {
    "Company": [
        "Goldman Sachs",  "Goldman Sachs",    "Goldman Sachs",
        "Bayview",        "Bayview",          "Bayview",
        "Delta",          "Delta",            "Delta"
    ],
    "Analysis Type": [
        "Skill Gap", "Keyword Alignment", "Fit Summary",
        "Skill Gap", "Keyword Alignment", "Fit Summary",
        "Skill Gap", "Keyword Alignment", "Fit Summary"
    ],
    # Fill these in after reviewing Cell 21 output:
    "Retrieval Relevance":  ["Yes"]*9,       # Yes / Partial / No
    "Skill ID Accuracy":    ["?/? correct"]*9,
    "Actionability (1-5)":  [0]*9,
    "Faithfulness":         ["Faithful"]*9   # Faithful / Partial / Hallucinated
}

eval_df = pd.DataFrame(eval_data)
display(eval_df)

,Company,Analysis Type,Retrieval Relevance,Skill ID Accuracy,Actionability (1-5),Faithfulness
0,Goldman Sachs,Skill Gap,Yes,?/? correct,0,Faithful
1,Goldman Sachs,Keyword Alignment,Yes,?/? correct,0,Faithful
2,Goldman Sachs,Fit Summary,Yes,?/? correct,0,Faithful
3,Bayview,Skill Gap,Yes,?/? correct,0,Faithful
4,Bayview,Keyword Alignment,Yes,?/? correct,0,Faithful
5,Bayview,Fit Summary,Yes,?/? correct,0,Faithful
6,Delta,Skill Gap,Yes,?/? correct,0,Faithful
7,Delta,Keyword Alignment,Yes,?/? correct,0,Faithful
8,Delta,Fit Summary,Yes,?/? correct,0,Faithful


### Evaluation Analysis

**Which analysis type worked best?**

**Which JDs produced the best/worst results? Why?**

**Where did the system hallucinate or produce inaccurate results?**

**What would you improve?**

---

## Next Steps

1. Build your Streamlit app (`streamlit_app.py`) using the pipeline from this notebook
2. Write your Technical Manager Memo (`memo.md`)
3. Complete your AI Usage Log (`ai_log.md`)
4. Verify GitHub repository structure and commit count

---
*BSAN 6200 | Spring 2026 | Assignment 5 — Option B*

In [37]:
import json

notebook_path = "/content/drive/MyDrive/Colab Notebooks/Assignment_5_LLM.ipynb"

with open(notebook_path, "r") as f:
    nb = json.load(f)

if "widgets" in nb.get("metadata", {}):
    del nb["metadata"]["widgets"]

with open(notebook_path, "w") as f:
    json.dump(nb, f, indent=1)

print("Done.")

Done.
